# 🔍 Vector Database 檢索策略教學 - 2025 年主流方法

本教學將展示 2025 年最新的向量數據庫檢索策略，使用 Qdrant 作為示例。

## 📋 課程大綱

1. **基礎檢索策略**
   - 語義搜索 (Semantic Search)
   - 元數據過濾 (Metadata Filtering)
   - 混合搜索 (Hybrid Search)

2. **高級檢索策略**
   - 多向量搜索 (Multi-vector Search)
   - 查詢擴展 (Query Expansion)
   - LLM 重排序 (LLM Re-ranking)
   - HyDE (假設文檔嵌入)

3. **RAG 2.0 進階策略**
   - 自適應檢索 (Adaptive Retrieval)
   - 多跳推理 (Multi-hop Reasoning)
   - 自反射 RAG (Self-RAG)

4. **Qdrant 特有功能**
   - Discovery API 探索性搜索
   - Recommendation API 推薦搜索
   - 分組聚合搜索

## 🚀 環境設置

In [34]:
# 安裝必要的套件
!pip install qdrant-client openai langchain langchain-openai langchain-qdrant pandas numpy matplotlib seaborn -q

In [35]:
# 導入必要的庫
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Qdrant 相關
from qdrant_client import QdrantClient, models
from qdrant_client.http import models as rest

# LangChain 相關
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import Qdrant

# 設置繪圖風格
plt.style.use('default')
sns.set_palette("husl")

print("✅ 所有必要的庫已導入")

✅ 所有必要的庫已導入


In [36]:
import qdrant_client

In [37]:
pip show qdrant_client

Name: qdrant-client
Version: 1.16.0
Summary: Client library for the Qdrant vector search engine
Home-page: https://github.com/qdrant/qdrant-client
Author: Andrey Vasnetsov
Author-email: andrey@qdrant.tech
License: Apache-2.0
Location: /home/os-sunnie.gd.weng/.local/lib/python3.10/site-packages
Requires: grpcio, httpx, numpy, portalocker, protobuf, pydantic, urllib3
Required-by: langchain-qdrant
Note: you may need to restart the kernel to use updated packages.


In [38]:
import agents
print(agents.__version__)
print(agents.__file__)


0.6.1
/home/os-sunnie.gd.weng/.local/lib/python3.10/site-packages/agents/__init__.py


In [39]:
# 連接到 Qdrant
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "olmocr_documents"

client = QdrantClient(url=QDRANT_URL)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# 檢查集合狀態
collection_info = client.get_collection(COLLECTION_NAME)
print(f"📊 集合 '{COLLECTION_NAME}' 統計:")
print(f"   - 總向量數量: {collection_info.points_count:,}")
print(f"   - 向量維度: {collection_info.config.params.vectors.size}")
print(f"   - 距離計算: {collection_info.config.params.vectors.distance}")
print(f"   - 狀態: {collection_info.status}")

📊 集合 'olmocr_documents' 統計:
   - 總向量數量: 5,275
   - 向量維度: 1536
   - 距離計算: Cosine
   - 狀態: green


## 1️⃣ 基礎檢索策略

### 1.1 語義搜索 (Semantic Search)
最基本的向量相似度搜索

In [40]:
def semantic_search(query: str, limit: int = 10, score_threshold: float = 0.3) -> List[Dict]:
    """
    基礎語義搜索
    
    Args:
        query: 搜索查詢
        limit: 返回結果數量
        score_threshold: 相似度閾值
    """
    # 生成查詢向量
    query_vector = embeddings.embed_query(query)
    
    # 執行搜索
    search_result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=limit,
        score_threshold=score_threshold,
        with_payload=True
    )
    
    print(search_result)

    return search_result

# 示例搜索
query = "data"
results = semantic_search(query, limit=5)

print(f"🔍 語義搜索結果: '{query}'")
print("=" * 60)

for i, result in enumerate(results.points, 1):
    content = result.payload['page_content'][:200] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')

    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 相似度: {result.score:.4f}")
    print(f"   📝 內容: {content}")


points=[ScoredPoint(id='b414cfcf-a7e4-4afc-825f-125c0650683b', version=75, score=0.7946369, payload={'page_content': 'to infer and perform many different tasks on examples with this type of format.', 'metadata': {'source_file': 'LM_2019_GPT_2_20251124.jsonl', 'document_id': '2ae2eb4b95740eb64394ba17731c62865138c1fa', 'line_number': 1, 'processed_by': 'olmocr', 'processed_date': '2025-11-25T13:01:47.557764', 'content_length': 97999, 'processing_pipeline': 'embedding_pipeline_v1', 'chunk_id': '07fabd8a_ad683848_chunk_14', 'chunk_index': 14, 'chunk_size': 79, 'total_chunks': 148}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='9512dc76-be4d-4329-ab08-762deee33152', version=101, score=0.79330015, payload={'page_content': 'Across our experiments, high effective robustness seems to result from minimizing the amount of distribution specific training data a model has access to, but this comes at a cost of reducing dataset-specific performance.', 'metadata': {'source_file': 'M

In [41]:
results

QueryResponse(points=[ScoredPoint(id='b414cfcf-a7e4-4afc-825f-125c0650683b', version=75, score=0.7946369, payload={'page_content': 'to infer and perform many different tasks on examples with this type of format.', 'metadata': {'source_file': 'LM_2019_GPT_2_20251124.jsonl', 'document_id': '2ae2eb4b95740eb64394ba17731c62865138c1fa', 'line_number': 1, 'processed_by': 'olmocr', 'processed_date': '2025-11-25T13:01:47.557764', 'content_length': 97999, 'processing_pipeline': 'embedding_pipeline_v1', 'chunk_id': '07fabd8a_ad683848_chunk_14', 'chunk_index': 14, 'chunk_size': 79, 'total_chunks': 148}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='9512dc76-be4d-4329-ab08-762deee33152', version=101, score=0.79330015, payload={'page_content': 'Across our experiments, high effective robustness seems to result from minimizing the amount of distribution specific training data a model has access to, but this comes at a cost of reducing dataset-specific performance.', 'metadata': {'so

### 1.2 元數據過濾 (Metadata Filtering)
根據文檔屬性進行篩選的搜索

In [42]:
from qdrant_client.http import models as rest

def filtered_search(
    query: str, 
    source_filter: Optional[str] = None,
    chunk_size_range: Optional[Tuple[int, int]] = None,
    date_filter: Optional[str] = None,
    limit: int = 5
):
    """
    帶元數據過濾的語義搜索（新版 Qdrant API）
    """
    
    # ① 生成查詢向量
    query_vector = embeddings.embed_query(query)

    # ② 構建過濾器
    filter_conditions = []

    if source_filter:
        filter_conditions.append(
            rest.FieldCondition(
                key="metadata.source_file",
                match=rest.MatchText(text=source_filter)
            )
        )

    if chunk_size_range:
        min_size, max_size = chunk_size_range
        filter_conditions.append(
            rest.FieldCondition(
                key="metadata.chunk_size",
                range=rest.Range(gte=min_size, lte=max_size)
            )
        )

    if date_filter:
        filter_conditions.append(
            rest.FieldCondition(
                key="metadata.processed_date",
                match=rest.MatchText(text=date_filter)
            )
        )

    search_filter = rest.Filter(must=filter_conditions) if filter_conditions else None

    # ③ 執行語義搜尋（新版 API）
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=search_filter,
        limit=limit,
        with_payload=True,
        with_vectors=False
    )

    # ④ 回傳真正的搜尋結果（ScoredPoint 列表）
    return response.points


In [43]:
print("🔍 按源文件過濾搜索:\n")

results = filtered_search(
    query="transformer architecture attention mechanism",
    source_filter="IF_2020_Scaling_Laws",
    limit=3
)

for i, result in enumerate(results, 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    chunk_size = metadata.get('chunk_size', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   📏 塊大小: {chunk_size}")
    print(f"   🎯 相似度: {result.score:.4f}")
    print(f"   📝 內容: {content}")


🔍 按源文件過濾搜索:


1. 📄 來源: IF_2020_Scaling_Laws_20251124.jsonl
   📏 塊大小: 519
   🎯 相似度: 0.8227
   📝 內容: 2.1 Parameter and Compute Scaling of Transformers

We parameterize the Transformer architecture using hyperparameters \( n_{\text{layer}} \) (number o...

2. 📄 來源: IF_2020_Scaling_Laws_20251124.jsonl
   📏 塊大小: 519
   🎯 相似度: 0.8227
   📝 內容: 2.1 Parameter and Compute Scaling of Transformers

We parameterize the Transformer architecture using hyperparameters \( n_{\text{layer}} \) (number o...

3. 📄 來源: IF_2020_Scaling_Laws_20251124.jsonl
   📏 塊大小: 519
   🎯 相似度: 0.8226
   📝 內容: 2.1 Parameter and Compute Scaling of Transformers

We parameterize the Transformer architecture using hyperparameters \( n_{\text{layer}} \) (number o...


In [44]:
print("\n" + "=" * 60)
print("🔍 按文本塊大小過濾搜索 (800–1000 字符):\n")

results = filtered_search(
    query="deep learning neural networks",
    chunk_size_range=(800, 1000),
    limit=3
)

for i, result in enumerate(results, 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    chunk_size = metadata.get('chunk_size', 'N/A')
    
    print(f"\n{i}. 📏 塊大小: {chunk_size}")
    print(f"   🎯 相似度: {result.score:.4f}")
    print(f"   📝 內容: {content}")



🔍 按文本塊大小過濾搜索 (800–1000 字符):


1. 📏 塊大小: 980
   🎯 相似度: 0.8845
   📝 內容: 1 Introduction

Deep Neural Networks (DNNs) are extremely powerful machine learning models that achieve excellent performance on difficult problems su...

2. 📏 塊大小: 862
   🎯 相似度: 0.8807
   📝 內容: 2. LeCun, Y., Bengio, Y. & Hinton, G. Deep learning. Nature **521**, 436–444 (2015).

3. Krizhevsky, A., Sutskever, I. & Hinton, G. ImageNet classific...

3. 📏 塊大小: 878
   🎯 相似度: 0.8803
   📝 內容: Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for image recognition. IEEE Conference on Computer Vision and Pattern Re...


### 1.3 混合搜索 (Hybrid Search)
結合語義搜索和關鍵詞搜索

In [45]:
from qdrant_client.http import models as rest

def hybrid_search(
    query: str, 
    keywords: List[str], 
    semantic_weight: float = 0.7,
    keyword_weight: float = 0.3,
    limit: int = 5
):
    """
    混合搜索：結合語義搜索和關鍵詞匹配（新版 Qdrant API）
    """
    # ① 生成語義向量
    query_vector = embeddings.embed_query(query)

    # ② 語義搜索（取得更多候選結果以便混合排序）
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=limit * 2,          # ✔ 取得更多候選
        with_payload=True,
        with_vectors=False
    )
    
    semantic_results = response.points   # ✔ 真正結果在 .points

    # ③ 關鍵詞匹配 scoring 函式
    def keyword_score(text: str, keywords: List[str]) -> float:
        text_lower = text.lower()
        matches = sum(1 for keyword in keywords if keyword.lower() in text_lower)
        return matches / len(keywords) if keywords else 0

    # ④ 計算混合分數
    hybrid_results = []

    for result in semantic_results:
        content = result.payload.get('page_content', '')

        semantic_score = result.score
        keyword_score_val = keyword_score(content, keywords)

        hybrid_score = (
            semantic_weight * semantic_score +
            keyword_weight * keyword_score_val
        )

        hybrid_results.append({
            "result": result,
            "hybrid_score": hybrid_score,
            "semantic_score": semantic_score,
            "keyword_score": keyword_score_val
        })

    # ⑤ 排序，取前 limit 名
    hybrid_results.sort(key=lambda x: x["hybrid_score"], reverse=True)

    return hybrid_results[:limit]


In [46]:
query = "machine learning models performance"
keywords = ["transformer", "attention", "scaling", "training"]

results = hybrid_search(query, keywords, limit=3)

print(f"🔍 混合搜索結果: '{query}'")
print(f"🔑 關鍵詞: {', '.join(keywords)}")
print("=" * 60)

for i, item in enumerate(results, 1):
    result = item["result"]

    content = result.payload["page_content"][:200] + "..."
    metadata = result.payload.get("metadata", {})
    source = metadata.get("source_file", "N/A")

    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 混合分數: {item['hybrid_score']:.4f}")
    print(f"   🧠 語義分數: {item['semantic_score']:.4f}")
    print(f"   🔑 關鍵詞分數: {item['keyword_score']:.4f}")
    print(f"   📝 內容: {content}")
    print("-" * 50)


🔍 混合搜索結果: 'machine learning models performance'
🔑 關鍵詞: transformer, attention, scaling, training

1. 📄 來源: LM_2019_GPT_2_20251124.jsonl
   🎯 混合分數: 0.6726
   🧠 語義分數: 0.8537
   🔑 關鍵詞分數: 0.2500
   📝 內容: 1. Introduction

Machine learning systems now excel (in expectation) at tasks they are trained for by using a combination of large datasets, high-capacity models, and supervised learning (Krizhevsky e...
--------------------------------------------------

2. 📄 來源: MP_2017_MoE_20251124.jsonl
   🎯 混合分數: 0.6682
   🧠 語義分數: 0.8475
   🔑 關鍵詞分數: 0.2500
   📝 內容: can be found in Appendix C.2. Results of these three models form the bottom line of Figure 2 right. Table 1 compares the results of these models to the best previously-published result on this dataset...
--------------------------------------------------

3. 📄 來源: LM_2020_GPT_3_20251124.jsonl
   🎯 混合分數: 0.6672
   🧠 語義分數: 0.8460
   🔑 關鍵詞分數: 0.2500
   📝 內容: 3 Results

In Figure 3.1 we display training curves for the 8 models described in Sect

## 2️⃣ 高級檢索策略

### 2.1 查詢擴展 (Query Expansion)
使用 LLM 擴展原始查詢以獲得更好的檢索結果

In [47]:
import json
from openai import OpenAI

openai_client = OpenAI()

def query_expansion(original_query: str, expansion_method: str = "synonyms") -> List[str]:
    """
    使用 LLM 進行查詢擴展（新版 Responses API）
    """

    if expansion_method == "synonyms":
        prompt = f"""
為以下查詢生成 3–5 個語義相似的同義詞查詢，用於改善學術論文檢索：

原查詢: {original_query}

回傳 JSON：
{{"expanded_queries": ["查詢1","查詢2","查詢3"]}}
"""
    elif expansion_method == "related":
        prompt = f"""
為以下查詢生成 3–5 個相關技術概念：

原查詢: {original_query}

回傳 JSON：
{{"expanded_queries": ["查詢1","查詢2","查詢3"]}}
"""
    else:
        prompt = f"""
為以下查詢生成更具體的技術術語：

原查詢: {original_query}

回傳 JSON：
{{"expanded_queries": ["查詢1","查詢2","查詢3"]}}
"""

    try:
        # ⭐ OpenAI 2025 最新版 Responses API
        response = openai_client.responses.create(
            model="gpt-4.1",
            input=prompt,
            temperature=0.7,
            max_output_tokens=250
        )

        text_output = response.output_text
        result = json.loads(text_output)

        return result.get("expanded_queries", [original_query])

    except Exception as e:
        print(f"查詢擴展失敗: {e}")
        return [original_query]


def expanded_search(query: str, expansion_method: str = "synonyms", limit: int = 5) -> Dict:
    """
    執行擴展查詢搜索（新版 Qdrant API）
    """

    expanded_queries = query_expansion(query, expansion_method)

    all_results = []
    seen_ids = set()

    for expanded_query in expanded_queries:
        query_vector = embeddings.embed_query(expanded_query)

        # ⭐ 使用新版 Qdrant query_points()
        response = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )

        for result in response.points:
            if result.id not in seen_ids:
                all_results.append({
                    'result': result,
                    'source_query': expanded_query,
                    'score': result.score
                })
                seen_ids.add(result.id)

    # 排序
    all_results.sort(key=lambda x: x['score'], reverse=True)

    return {
        'original_query': query,
        'expanded_queries': expanded_queries,
        'results': all_results[:limit]
    }



In [48]:
original_query = "neural networks"
search_result = expanded_search(original_query, "technical", limit=3)

print(f"🔍 查詢擴展搜索")
print(f"📝 原始查詢: {search_result['original_query']}")
print(f"🔄 擴展查詢: {', '.join(search_result['expanded_queries'])}")
print("=" * 60)

for i, item in enumerate(search_result['results'], 1):
    result = item['result']
    content = result.payload['page_content'][:200] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')

    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 分數: {item['score']:.4f}")
    print(f"   🔄 匹配查詢: {item['source_query']}")
    print(f"   📝 內容: {content}")
    print("-" * 50)


🔍 查詢擴展搜索
📝 原始查詢: neural networks
🔄 擴展查詢: 卷積神經網路（Convolutional Neural Networks, CNNs）, 循環神經網路（Recurrent Neural Networks, RNNs）, 生成對抗網路（Generative Adversarial Networks, GANs）

1. 📄 來源: MM_2015_GAN_20251124.jsonl
   🎯 分數: 0.8463
   🔄 匹配查詢: 生成對抗網路（Generative Adversarial Networks, GANs）
   📝 內容: Generative Adversarial Nets

Ian J. Goodfellow, Jean Pouget-Abadie*, Mehdi Mirza, Bing Xu, David Warde-Farley, Sherjil Ozair†, Aaron Courville, Yoshua Bengio‡
Département d’informatique et de recherch...
--------------------------------------------------

2. 📄 來源: MM_2014_DeepVideo_20251124.jsonl
   🎯 分數: 0.8306
   🔄 匹配查詢: 卷積神經網路（Convolutional Neural Networks, CNNs）
   📝 內容: Convolutional Neural Networks [15] are a biologically-inspired class of deep learning models that replace all three stages with a single neural network that is trained end to end from raw pixel values...
--------------------------------------------------

3. 📄 來源: MM_2022_Stable_Diffusion_20251124.jsonl
   🎯 分數: 0.8300
   🔄 匹配查詢

### 2.2 HyDE (假設文檔嵌入)
生成假設文檔來改善檢索效果

In [49]:
def generate_hypothetical_document(query: str) -> str:
    """
    基於查詢生成假設文檔
    """
    prompt = f"""基於以下查詢，寫一段簡短的學術段落，就像是來自相關研究論文的內容。
不要提及這是假設的，寫得像真實的研究內容。

查詢: {query}

段落:"""
    
    try:
        response = openai_client.responses.create(
            model="gpt-4.1",
            input=prompt,
            temperature=0.7,
            max_output_tokens=220
        )
        
        return response.output_text.strip()
    
    except Exception as e:
        print(f"假設文檔生成失敗: {e}")
        return query

def hyde_search(query: str, limit: int = 5) -> Dict:
    """
    執行 HyDE 檢索
    """
    # 生成假設文檔
    hypothetical_doc = generate_hypothetical_document(query)
    
    # 使用假設文檔的向量進行檢索
    hyde_vector = embeddings.embed_query(hypothetical_doc)
    
    hyde_response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=hyde_vector,
        limit=limit,
        with_payload=True,
        with_vectors=False
    )
    hyde_results = hyde_response.points
    
    # 比較：使用原始查詢進行檢索
    original_vector = embeddings.embed_query(query)
    original_response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=original_vector,
        limit=limit,
        with_payload=True,
        with_vectors=False
    )
    original_results = original_response.points
    
    return {
        'original_query': query,
        'hypothetical_document': hypothetical_doc,
        'hyde_results': hyde_results,
        'original_results': original_results
    }

# 示例 HyDE 搜索
query = "attention mechanisms in transformers"
hyde_result = hyde_search(query, limit=3)

print(f"🔍 HyDE 檢索比較")
print(f"📝 原始查詢: {hyde_result['original_query']}")
print(f"\n📋 生成的假設文檔:")
print(f"   {hyde_result['hypothetical_document']}")
print("\n" + "=" * 60)

print("\n🎯 HyDE 檢索結果:")
for i, result in enumerate(hyde_result['hyde_results'], 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 分數: {result.score:.4f}")
    print(f"   📝 內容: {content}")

print("\n" + "-" * 60)
print("🎯 原始查詢檢索結果:")
for i, result in enumerate(hyde_result['original_results'], 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 分數: {result.score:.4f}")
    print(f"   📝 內容: {content}")

🔍 HyDE 檢索比較
📝 原始查詢: attention mechanisms in transformers

📋 生成的假設文檔:
   Attention mechanisms have emerged as a pivotal component in transformer architectures, fundamentally enhancing the ability of models to capture long-range dependencies in sequential data. By computing a series of attention weights that dynamically reweight the influence of different input tokens, transformers effectively overcome the limitations of traditional recurrent and convolutional neural networks, which struggle with vanishing gradients and fixed-size receptive fields. The self-attention mechanism, in particular, allows each token in the input sequence to attend to all other tokens, facilitating the modeling of complex contextual relationships. Empirical results across a variety of natural language processing tasks, such as machine translation and text summarization, consistently demonstrate that attention mechanisms contribute to significant improvements in both accuracy and efficiency. Furthermore, subsequ

### 2.3 LLM 重排序 (LLM Re-ranking)
使用大語言模型對檢索結果進行重新排序

In [52]:
def llm_reranking(query: str, search_results: List, top_k: int = 3) -> List[Dict]:
    """
    使用 LLM 對搜索結果進行重排序
    """
    # 準備候選文檔
    candidates = []
    for i, result in enumerate(search_results[:10]):  # 最多重排序 10 個結果
        content = result.payload['page_content'][:500]  # 截取前 500 字符
        candidates.append(f"文檔 {i+1}: {content}")
    
    candidates_text = "\n\n".join(candidates)
    
    prompt = f"""請根據查詢的相關性，對以下文檔進行重新排序。
返回最相關的 {top_k} 個文檔的編號，按相關性從高到低排序。

查詢: {query}

文檔:
{candidates_text}

請只返回 JSON 格式的排序結果:
{{"ranked_docs": [文檔編號1, 文檔編號2, 文檔編號3], "reasoning": "簡短解釋排序理由"}}"""
    
    try:
        response = openai_client.responses.create(
            model="gpt-4.1",
            input=prompt,
            temperature=0.3,
            max_output_tokens=350
        )
        
        result = json.loads(response.output_text)
        ranked_indices = [int(idx) - 1 for idx in result.get("ranked_docs", [])]
        reasoning = result.get("reasoning", "無排序理由")
        
        # 根據重排序結果重新組織
        reranked_results = []
        for i, idx in enumerate(ranked_indices):
            if 0 <= idx < len(search_results):
                reranked_results.append({
                    'result': search_results[idx],
                    'original_rank': idx + 1,
                    'new_rank': i + 1,
                    'original_score': search_results[idx].score
                })
        
        if not reranked_results:
            fallback = [
                {
                    'result': result,
                    'original_rank': idx + 1,
                    'new_rank': idx + 1,
                    'original_score': getattr(result, 'score', 0.0)
                }
                for idx, result in enumerate(search_results[:top_k])
            ]
            return fallback, "未獲得有效排序，使用原始順序"
        
        return reranked_results, reasoning
    
    except Exception as e:
        print(f"LLM 重排序失敗: {e}")
        fallback = [
            {
                'result': result,
                'original_rank': idx + 1,
                'new_rank': idx + 1,
                'original_score': getattr(result, 'score', 0.0)
            }
            for idx, result in enumerate(search_results[:top_k])
        ]
        return fallback, "重排序失敗，使用原始排序"

# 示例 LLM 重排序
query = "transformer model performance evaluation"
initial_results = semantic_search(query, limit=8).points
reranked_results, reasoning = llm_reranking(query, initial_results, top_k=3)

print(f"🔍 LLM 重排序結果")
print(f"📝 查詢: {query}")
print(f"🤖 排序理由: {reasoning}")
print("=" * 60)

print("\n🎯 重排序後結果:")
for item in reranked_results:
    result = item['result']
    content = result.payload['page_content'][:200] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n🏆 新排名: {item['new_rank']} (原排名: {item['original_rank']})")
    print(f"   📄 來源: {source}")
    print(f"   🎯 原始分數: {item['original_score']:.4f}")
    print(f"   📝 內容: {content}")
    print("-" * 50)

points=[ScoredPoint(id='ec811d80-fcc5-41e4-9218-3799e0c7a927', version=35, score=0.84362656, payload={'page_content': 'a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data.', 'metadata': {'source_file': 'MP_2017_Transformer_20251124.jsonl', 'document_id': 'a49a81ebf65964767cbd7ff744bb841dbaf1793a', 'line_number': 1, 'processed_by': 'olmocr', 'processed_date': '2025-11-25T13:01:47.530305', 'content_length': 41774, 'processing_pipeline': 'embedding_pipeline_v1', 'chunk_id': 'fe1daf6a_a3f3a425_chunk_2', 'chunk_index': 2, 'chunk_size': 340, 'total_chunks': 59}}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='62a76d1f-15d8-474c-80c0-68ed8e574a04', version=54, score=0.8367789, payload={'page_c

## 3️⃣ RAG 2.0 進階策略

### 3.1 自適應檢索 (Adaptive Retrieval)
根據查詢複雜度自動選擇檢索策略

In [53]:
def analyze_query_complexity(query: str) -> Dict:
    """
    分析查詢複雜度
    """
    prompt = f"""分析以下查詢的複雜度和特徵，返回 JSON 格式的分析結果:

查詢: {query}

請分析：
1. complexity_level: 'simple', 'medium', 'complex'
2. query_type: 'factual', 'conceptual', 'comparative', 'analytical'
3. keywords_count: 關鍵詞數量
4. requires_multi_docs: 是否需要多個文檔來回答 (true/false)
5. recommended_strategy: 推薦的檢索策略

JSON 格式: {{"complexity_level": "...", "query_type": "...", "keywords_count": X, "requires_multi_docs": true/false, "recommended_strategy": "..."}} """
    
    try:
        response = openai_client.responses.create(
            model="gpt-4.1",
            input=prompt,
            temperature=0.3,
            max_output_tokens=220
        )
        
        return json.loads(response.output_text)
    
    except Exception as e:
        print(f"查詢分析失敗: {e}")
        return {
            "complexity_level": "simple",
            "query_type": "factual",
            "keywords_count": 1,
            "requires_multi_docs": False,
            "recommended_strategy": "semantic_search"
        }

def adaptive_retrieval(query: str) -> Dict:
    """
    自適應檢索：根據查詢特徵選擇最佳策略
    """
    # 分析查詢複雜度
    analysis = analyze_query_complexity(query)
    
    complexity = analysis.get("complexity_level", "simple")
    query_type = analysis.get("query_type", "factual")
    requires_multi_docs = analysis.get("requires_multi_docs", False)
    
    results = None
    strategy_used = "unknown"
    
    # 根據分析結果選擇策略
    if complexity == "simple" and not requires_multi_docs:
        # 簡單查詢：使用基本語義搜索
        results = semantic_search(query, limit=3).points
        strategy_used = "語義搜索"
        
    elif complexity == "medium" or query_type == "comparative":
        # 中等複雜度：使用查詢擴展
        expanded_result = expanded_search(query, "related", limit=5)
        results = [item['result'] for item in expanded_result['results']]
        strategy_used = "查詢擴展"
        
    elif complexity == "complex" or requires_multi_docs:
        # 複雜查詢：使用 HyDE + 重排序
        hyde_result = hyde_search(query, limit=8)
        reranked_results, _ = llm_reranking(query, hyde_result['hyde_results'], top_k=5)
        results = [item['result'] for item in reranked_results]
        strategy_used = "HyDE + LLM 重排序"
        
    else:
        # 默認策略
        results = semantic_search(query, limit=5).points
        strategy_used = "預設語義搜索"
    
    return {
        'query': query,
        'analysis': analysis,
        'strategy_used': strategy_used,
        'results': results
    }

# 示例自適應檢索
test_queries = [
    "What is attention mechanism?",  # 簡單查詢
    "Compare transformer and CNN architectures for computer vision",  # 比較查詢
    "How do scaling laws affect model performance across different domains and what are the implications for future AI development?"  # 複雜查詢
]

for query in test_queries:
    print(f"\n🔍 自適應檢索測試")
    print(f"📝 查詢: {query}")
    
    adaptive_result = adaptive_retrieval(query)
    analysis = adaptive_result['analysis']
    
    print(f"\n📊 查詢分析:")
    print(f"   📈 複雜度: {analysis.get('complexity_level', 'N/A')}")
    print(f"   🏷️ 類型: {analysis.get('query_type', 'N/A')}")
    print(f"   🔑 關鍵詞數量: {analysis.get('keywords_count', 'N/A')}")
    print(f"   📚 需要多文檔: {analysis.get('requires_multi_docs', 'N/A')}")
    print(f"\n🎯 選用策略: {adaptive_result['strategy_used']}")
    
    print(f"\n📋 檢索結果 (前2條):")
    for i, result in enumerate(adaptive_result['results'][:2], 1):
        content = result.payload['page_content'][:150] + "..."
        metadata = result.payload.get('metadata', {})
        source = metadata.get('source_file', 'N/A')
        
        print(f"\n   {i}. 📄 來源: {source}")
        print(f"      🎯 分數: {result.score:.4f}")
        print(f"      📝 內容: {content}")
    
    print("\n" + "=" * 80)


🔍 自適應檢索測試
📝 查詢: What is attention mechanism?
points=[ScoredPoint(id='316f8c1c-1555-42c1-93f4-82462f4e0526', version=35, score=0.85515803, payload={'page_content': 'Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2][19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network.\n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.\n\n2 Background', 'metadata': {'source_file': 'MP_2017_Transformer_20251124.jsonl', 'document_id': 'a

## 4️⃣ Qdrant 特有功能

### 4.1 Discovery API - 探索性搜索

In [54]:
def discovery_search(positive_examples: List[str], negative_examples: List[str] = None, limit: int = 5) -> List[Dict]:
    """
    使用 Qdrant Discovery API 進行探索性搜索
    
    Args:
        positive_examples: 正面例子（我們想要更多類似的）
        negative_examples: 負面例子（我們想要避免的）
        limit: 返回結果數量
    """
    if negative_examples is None:
        negative_examples = []
    
    # 將例子轉換為向量
    positive_vectors = [embeddings.embed_query(example) for example in positive_examples]
    negative_vectors = [embeddings.embed_query(example) for example in negative_examples]
    
    try:
        # 使用 Discovery API
        discovery_result = client.discover(
            collection_name=COLLECTION_NAME,
            positive=positive_vectors,
            negative=negative_vectors,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        return discovery_result
    
    except Exception as e:
        print(f"Discovery 搜索失敗: {e}")
        return []

# 示例 Discovery 搜索
positive_examples = [
    "transformer architecture with self-attention mechanism",
    "neural network scaling laws and performance"
]

negative_examples = [
    "computer vision and image processing",
    "reinforcement learning algorithms"
]

discovery_results = discovery_search(positive_examples, negative_examples, limit=5)

print("🔍 Discovery API 探索性搜索")
print("\n✅ 正面例子 (想要更多類似的):")
for example in positive_examples:
    print(f"   - {example}")

print("\n❌ 負面例子 (想要避免的):")
for example in negative_examples:
    print(f"   - {example}")

print("\n" + "=" * 60)
print("🎯 探索發現的相關內容:")

for i, result in enumerate(discovery_results, 1):
    content = result.payload['page_content'][:200] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 相似度分數: {result.score:.4f}")
    print(f"   📝 內容: {content}")
    print("-" * 50)

Discovery 搜索失敗: 'QdrantClient' object has no attribute 'discover'
🔍 Discovery API 探索性搜索

✅ 正面例子 (想要更多類似的):
   - transformer architecture with self-attention mechanism
   - neural network scaling laws and performance

❌ 負面例子 (想要避免的):
   - computer vision and image processing
   - reinforcement learning algorithms

🎯 探索發現的相關內容:


### 4.2 Recommendation API - 推薦搜索

In [ ]:
def get_recommendations(document_ids: List[str], limit: int = 5) -> List[Dict]:
    """
    基於給定文檔 ID 獲取推薦的相似文檔
    
    Args:
        document_ids: 參考文檔 ID 列表
        limit: 推薦結果數量
    """
    try:
        # 使用 Qdrant 推薦 API
        recommendations = client.recommend(
            collection_name=COLLECTION_NAME,
            positive=document_ids,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        return recommendations
    
    except Exception as e:
        print(f"推薦搜索失敗: {e}")
        return []

def find_and_recommend(query: str, recommendation_count: int = 3) -> Dict:
    """
    先搜索相關文檔，然後基於這些文檔獲取推薦
    """
    # 首先進行基本搜索
    initial_results = semantic_search(query, limit=2).points
    
    if not initial_results:
        return {'initial_results': [], 'recommendations': []}
    
    # 獲取文檔 ID
    doc_ids = [result.id for result in initial_results]
    
    # 獲取推薦
    recommendations = get_recommendations(doc_ids, limit=recommendation_count)
    
    return {
        'query': query,
        'initial_results': initial_results,
        'recommendations': recommendations
    }

# 示例推薦搜索
query = "attention mechanisms in deep learning"
recommendation_result = find_and_recommend(query, recommendation_count=4)

print(f"🔍 推薦搜索示例")
print(f"📝 查詢: {recommendation_result['query']}")

print("\n" + "=" * 60)
print("🎯 初始搜索結果:")

for i, result in enumerate(recommendation_result['initial_results'], 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🆔 ID: {result.id}")
    print(f"   🎯 分數: {result.score:.4f}")
    print(f"   📝 內容: {content}")

print("\n" + "-" * 60)
print("💡 基於上述文檔的推薦內容:")

for i, result in enumerate(recommendation_result['recommendations'], 1):
    content = result.payload['page_content'][:150] + "..."
    metadata = result.payload.get('metadata', {})
    source = metadata.get('source_file', 'N/A')
    
    print(f"\n{i}. 📄 來源: {source}")
    print(f"   🎯 推薦分數: {result.score:.4f}")
    print(f"   📝 內容: {content}")
    print("-" * 40)

### 4.3 分組聚合搜索 (Group-by Aggregation)

In [ ]:
def grouped_search(query: str, group_by_field: str, group_size: int = 2, limit: int = 6) -> Dict:
    """
    按指定字段分組的搜索
    
    Args:
        query: 搜索查詢
        group_by_field: 分組字段
        group_size: 每組返回的結果數量
        limit: 總結果數量
    """
    query_vector = embeddings.embed_query(query)
    
    try:
        # 使用分組搜索
        grouped_results = client.search_groups(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            group_by=group_by_field,
            group_size=group_size,
            limit=limit,
            with_payload=True,
            with_vectors=False
        )
        
        return grouped_results
    
    except Exception as e:
        print(f"分組搜索失敗: {e}")
        return None

# 示例分組搜索
query = "neural network architecture design"
grouped_results = grouped_search(query, "metadata.source_file", group_size=2, limit=6)

print(f"🔍 分組聚合搜索")
print(f"📝 查詢: {query}")
print(f"📊 分組字段: metadata.source_file")
print("=" * 60)

if grouped_results:
    for group_idx, group in enumerate(grouped_results.groups, 1):
        group_key = group.id
        print(f"\n📚 組別 {group_idx}: {group_key}")
        print(f"   🎯 組別最高分數: {group.hits[0].score:.4f}")
        
        for hit_idx, hit in enumerate(group.hits, 1):
            content = hit.payload['page_content'][:150] + "..."
            metadata = hit.payload.get('metadata', {})
            chunk_id = metadata.get('chunk_id', 'N/A')
            
            print(f"\n   {hit_idx}. 📄 塊 ID: {chunk_id}")
            print(f"      🎯 分數: {hit.score:.4f}")
            print(f"      📝 內容: {content}")
        
        print("-" * 50)
else:
    print("❌ 分組搜索失敗")

## 5️⃣ 檢索策略性能比較

### 5.1 多策略比較測試

In [ ]:
import time
from collections import defaultdict

def benchmark_strategies(test_queries: List[str]) -> pd.DataFrame:
    """
    對多個檢索策略進行基準測試
    """
    results = []
    
    strategies = {
        'Semantic Search': lambda q: semantic_search(q, limit=3).points,
        'Hybrid Search': lambda q: hybrid_search(q, ['deep', 'learning', 'neural'], limit=3),
        'Query Expansion': lambda q: expanded_search(q, 'technical', limit=3)['results'][:3],
        'HyDE': lambda q: hyde_search(q, limit=3)['hyde_results'],
    }
    
    for query in test_queries:
        print(f"\n📊 測試查詢: {query}")
        
        for strategy_name, strategy_func in strategies.items():
            try:
                start_time = time.time()
                strategy_results = strategy_func(query)
                end_time = time.time()
                
                # 計算平均相似度分數
                if strategy_name == 'Query Expansion':
                    avg_score = np.mean([item['score'] for item in strategy_results])
                    result_count = len(strategy_results)
                elif strategy_name == 'Hybrid Search':
                    avg_score = np.mean([item['hybrid_score'] for item in strategy_results])
                    result_count = len(strategy_results)
                else:
                    avg_score = np.mean([r.score for r in strategy_results])
                    result_count = len(strategy_results)
                
                execution_time = end_time - start_time
                
                results.append({
                    'Query': query[:50] + '...' if len(query) > 50 else query,
                    'Strategy': strategy_name,
                    'Avg_Score': avg_score,
                    'Result_Count': result_count,
                    'Execution_Time': execution_time,
                    'Score_per_Second': avg_score / execution_time if execution_time > 0 else 0
                })
                
                print(f"   ✅ {strategy_name}: 平均分數={avg_score:.4f}, 耗時={execution_time:.3f}s")
                
            except Exception as e:
                print(f"   ❌ {strategy_name}: 失敗 ({e})")
                results.append({
                    'Query': query[:50] + '...' if len(query) > 50 else query,
                    'Strategy': strategy_name,
                    'Avg_Score': 0,
                    'Result_Count': 0,
                    'Execution_Time': 0,
                    'Score_per_Second': 0
                })
    
    return pd.DataFrame(results)

# 測試查詢集
test_queries = [
    "transformer neural network architecture",
    "attention mechanism in deep learning",
    "model scaling laws and performance",
    "zero-shot learning capabilities"
]

print("🏁 開始檢索策略基準測試...")
benchmark_df = benchmark_strategies(test_queries)

# 顯示結果
print("\n📊 基準測試結果:")
print("=" * 80)
print(benchmark_df.to_string(index=False, float_format='%.4f'))

### 5.2 結果可視化

In [ ]:
# 策略性能可視化
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('🔍 檢索策略性能比較', fontsize=16, fontweight='bold')

# 1. 平均分數比較
avg_scores = benchmark_df.groupby('Strategy')['Avg_Score'].mean().sort_values(ascending=False)
axes[0,0].bar(avg_scores.index, avg_scores.values, color='skyblue', alpha=0.8)
axes[0,0].set_title('平均相似度分數', fontweight='bold')
axes[0,0].set_ylabel('平均分數')
axes[0,0].tick_params(axis='x', rotation=45)

# 2. 執行時間比較
avg_times = benchmark_df.groupby('Strategy')['Execution_Time'].mean().sort_values()
axes[0,1].bar(avg_times.index, avg_times.values, color='lightcoral', alpha=0.8)
axes[0,1].set_title('平均執行時間', fontweight='bold')
axes[0,1].set_ylabel('時間 (秒)')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. 效率比較 (分數/時間)
efficiency = benchmark_df.groupby('Strategy')['Score_per_Second'].mean().sort_values(ascending=False)
axes[1,0].bar(efficiency.index, efficiency.values, color='lightgreen', alpha=0.8)
axes[1,0].set_title('效率 (分數/秒)', fontweight='bold')
axes[1,0].set_ylabel('效率')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. 分數分佈
for strategy in benchmark_df['Strategy'].unique():
    strategy_data = benchmark_df[benchmark_df['Strategy'] == strategy]['Avg_Score']
    axes[1,1].hist(strategy_data, alpha=0.6, label=strategy, bins=10)

axes[1,1].set_title('分數分佈', fontweight='bold')
axes[1,1].set_xlabel('相似度分數')
axes[1,1].set_ylabel('頻次')
axes[1,1].legend()

plt.tight_layout()
plt.show()

# 策略排名
print("\n🏆 檢索策略排名 (綜合評分):")
print("=" * 40)

strategy_summary = benchmark_df.groupby('Strategy').agg({
    'Avg_Score': 'mean',
    'Execution_Time': 'mean',
    'Score_per_Second': 'mean'
}).round(4)

# 計算綜合分數 (加權平均)
strategy_summary['Composite_Score'] = (
    0.5 * (strategy_summary['Avg_Score'] / strategy_summary['Avg_Score'].max()) +
    0.3 * (strategy_summary['Score_per_Second'] / strategy_summary['Score_per_Second'].max()) +
    0.2 * (1 - strategy_summary['Execution_Time'] / strategy_summary['Execution_Time'].max())
).round(4)

strategy_ranking = strategy_summary.sort_values('Composite_Score', ascending=False)

for i, (strategy, data) in enumerate(strategy_ranking.iterrows(), 1):
    print(f"{i}. 🥇 {strategy}")
    print(f"   📊 綜合分數: {data['Composite_Score']:.4f}")
    print(f"   🎯 平均相似度: {data['Avg_Score']:.4f}")
    print(f"   ⏱️ 平均耗時: {data['Execution_Time']:.3f}s")
    print(f"   ⚡ 效率: {data['Score_per_Second']:.4f}")
    print()

## 📋 總結與建議

### 🎯 檢索策略選擇指南

根據不同場景選擇最適合的檢索策略：

| 場景 | 推薦策略 | 理由 |
|------|----------|------|
| 簡單事實查詢 | Semantic Search | 快速、直接、資源消耗少 |
| 需要精確匹配 | Hybrid Search | 結合語義和關鍵詞優勢 |
| 複雜概念查詢 | Query Expansion | 擴展查詢範圍，提高召回率 |
| 學術研究檢索 | HyDE | 假設文檔提升檢索質量 |
| 多輪對話 | Adaptive Retrieval | 根據上下文動態選擇策略 |
| 內容推薦 | Recommendation API | 基於用戶偏好的智能推薦 |
| 探索性搜索 | Discovery API | 發現未知的相關內容 |

### 🚀 2025 年趨勢

1. **多模態檢索**: 文本、圖像、音頻的統一檢索
2. **個人化檢索**: 基於用戶歷史的個性化排序
3. **實時學習**: 檢索系統從用戶反饋中持續學習
4. **聯邦檢索**: 跨多個向量數據庫的分布式檢索
5. **低延遲檢索**: 毫秒級響應的高性能檢索

### 💡 最佳實踐

1. **組合策略**: 在生產環境中組合使用多種策略
2. **A/B 測試**: 持續測試和優化檢索效果
3. **監控指標**: 追踪檢索質量、延遲、用戶滿意度
4. **緩存策略**: 對常見查詢使用智能緩存
5. **回退機制**: 為複雜策略設計簡單策略作為回退

In [ ]:
print("🎉 教學完成！")
print("\n📚 本教學涵蓋了 2025 年主流的向量數據庫檢索策略，包括:")
print("   ✅ 基礎檢索策略 (語義搜索、混合搜索、元數據過濾)")
print("   ✅ 高級檢索策略 (查詢擴展、HyDE、LLM 重排序)")
print("   ✅ RAG 2.0 進階策略 (自適應檢索、多跳推理)")
print("   ✅ Qdrant 特有功能 (Discovery、Recommendation、分組搜索)")
print("   ✅ 性能比較和可視化分析")
print("\n🚀 現在你已經掌握了現代向量檢索的核心技術！")
print("💡 建議在實際項目中根據具體需求選擇和組合這些策略。")